In [2]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

2025-09-25 16:13:17 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Subir inscritos

In [15]:
# Leer datos
df_inscritos = pd.read_excel('main_data/Base final feria EVA - inscripciones.xlsx', sheet_name='Hoja10')
df_all_terminals = pd.read_excel('main_data/EVA BANCOLOMBIA 1RA.xlsx')

In [17]:
# Subir a lz
sparky.subir_df(df_inscritos, 'mdo_adquirencia_feria_eva_inscritos', 'proceso_bluekai')

2025-09-22 20:25:22 - [INFO] - Intento 1 de 3


----------------------------------------------------------------------------------------
  i   tipo                 nombre                   estado     hora_inicio   duracion   
----------------------------------------------------------------------------------------
 6/6 DF A LZ mdo_adquirencia_feria_eva_inscritos   finalizado   08:25:22 PM     00:55.9 
----------------------------------------------------------------------------------------


In [14]:
# Subir a lz
sparky.subir_df(df_all_terminals, 'mdo_adquirencia_feria_eva_all_terminals', 'proceso_bluekai')

2025-09-19 11:34:14 - [INFO] - Intento 1 de 3


------------------------------------------------------------------------------------------------
  i     tipo                     nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------------
 4/4     DF A LZ mdo_adquirencia_feria_eva_all_terminals   finalizado   11:34:14 AM     01:05.3 
------------------------------------------------------------------------------------------------


# Obtener id_terminal adecuada

In [ ]:
# Hay muchos errores cuando se creo el archivo de inscritos, por tanto se hace una correción manual
# Se cambiará los datos de aquellos clientes que tengan trxs en el periodo 20250911 - 20250914
# Se buscará las terminales únicas de los inscritos que tienen trxs en el periodo de tiempo 20250911 - 20250914
# Luego se modificará manualmente la fuente de datos de inscritos
sql = """
WITH inscritos AS
  (SELECT *
   FROM proceso_bluekai.mdo_adquirencia_feria_eva_inscritos
   WHERE cu IS NOT NULL
     AND id_terminal IS NOT NULL
     AND pregunta IS NOT NULL)
select a.nombre_comercio, a.cod_unico, a.id_terminal, count(*)
FROM proceso_bluekai.mdo_adquirencia_trxs_20250911_20250914 as a
LEFT SEMI JOIN inscritos as b on cast(a.cod_unico as BIGINT) = cast(b.cu as BIGINT)
GROUP BY 1,2,3
ORDER BY a.nombre_comercio, a.cod_unico, a.id_terminal;
"""

# Obtener municipio de la adquirencia

In [4]:
dict_ult_ing_comercios = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_m_comercios')
dict_ult_ing_comercios

2025-09-25 16:18:02 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_m_comercios
/Users/santlond/anaconda3/envs/env_odbc_py39/lib/python3.9/site-packages/helper/helper.py:421: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-09-25 16:18:03 - [INFO] - Finalizo la busqueda, duracion: 00:00.9, resultado: {'year': 2025, 'month': 9, 'day': 25}


{'year': 2025, 'month': 9, 'day': 25}

In [6]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_gsap_m_comercios PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_adquirencia_gsap_m_comercios STORED AS PARQUET AS
select id_comercio, descri_ciudad, descri_dpto, descri_municipio, nit
from resultados_vspc_medios_de_pago.gsap_m_comercios
where year = """ + str(dict_ult_ing_comercios['year']) + """
and month = """ + str(dict_ult_ing_comercios['month']) + """
and day = """ + str(dict_ult_ing_comercios['day']) + """;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_gsap_m_comercios;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 7/7    DROP proceso.mdo_adquirencia_gsap_m_comercios   finalizado   04:18:56 PM     00:00.4 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 8/8  CREATE proceso.mdo_adquirencia_gsap_m_comercios   finalizado   04:18:57 PM     00:01.7 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

# Ganadores

In [8]:
# Ganadores por trxs
sql = """
WITH inscritos AS
  (SELECT nit,
          razon_social,
          marca,
          cu,
          id_terminal,
          pregunta
   FROM proceso_bluekai.mdo_adquirencia_feria_eva_inscritos
   WHERE cu IS NOT NULL
     AND id_terminal IS NOT NULL
     AND pregunta IS NOT NULL
   GROUP BY 1,
            2,
            3,
            4,
            5,
            6),
     trxs_inscritos AS
  (SELECT a.*
   FROM proceso_bluekai.mdo_adquirencia_trxs_20250911_20250914 AS a LEFT semi
   JOIN inscritos AS b ON cast(a.cod_unico AS BIGINT) = cast(b.cu AS BIGINT)
   AND cast(a.id_terminal AS VARCHAR) = cast(b.id_terminal AS VARCHAR)), agregados as (
SELECT cod_unico,
       nombre_comercio,
       count(*) AS trxs,
       sum(mnt_total_trx) AS fact
FROM trxs_inscritos
GROUP BY 1,
         2)
select a.*, b.descri_ciudad, b.descri_municipio, b.descri_dpto
FROM agregados as a
left join proceso.mdo_adquirencia_gsap_m_comercios as b on cast(a.cod_unico as BIGINT) = cast(b.id_comercio as BIGINT)
ORDER BY a.trxs DESC, a.fact DESC;
"""
helper.obtener_dataframe(sql).to_excel('main_data/ganadores_trxs_feria_eva_corte1.xlsx', index=False)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 13/13 DATAFRAME                                            ejecutando   04:32:11 PM             

2025-09-25 16:32:13 - [INFO] - 68 filas, 7 columnas, 00:01.9 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 13/13 DATAFRAME                                            finalizado   04:32:11 PM     00:02.0 
-------------------------------------------------------------------------------------------------


In [9]:
# Ganadores por facturación
sql = """
WITH inscritos AS
  (SELECT nit,
          razon_social,
          marca,
          cu,
          id_terminal,
          pregunta
   FROM proceso_bluekai.mdo_adquirencia_feria_eva_inscritos
   WHERE cu IS NOT NULL
     AND id_terminal IS NOT NULL
     AND pregunta IS NOT NULL
   GROUP BY 1,
            2,
            3,
            4,
            5,
            6),
     trxs_inscritos AS
  (SELECT a.*
   FROM proceso_bluekai.mdo_adquirencia_trxs_20250911_20250914 AS a LEFT semi
   JOIN inscritos AS b ON cast(a.cod_unico AS BIGINT) = cast(b.cu AS BIGINT)
   AND cast(a.id_terminal AS VARCHAR) = cast(b.id_terminal AS VARCHAR)), agregados as (
SELECT cod_unico,
       nombre_comercio,
       sum(mnt_total_trx) AS fact,
       count(*) AS trxs
FROM trxs_inscritos
GROUP BY 1,
         2)
select a.*, b.descri_ciudad, b.descri_municipio, b.descri_dpto
FROM agregados as a
left join proceso.mdo_adquirencia_gsap_m_comercios as b on cast(a.cod_unico as BIGINT) = cast(b.id_comercio as BIGINT)
order by a.fact DESC, a.trxs DESC;
"""
helper.obtener_dataframe(sql).to_excel('main_data/ganadores_facturacion_feria_eva_corte1.xlsx', index=False)


-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 14/14 DATAFRAME                                           descargando   04:32:15 PM             

2025-09-25 16:32:17 - [INFO] - 68 filas, 7 columnas, 00:01.6 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 14/14 DATAFRAME                                            finalizado   04:32:15 PM     00:01.8 
-------------------------------------------------------------------------------------------------


# Transacciones Adquirencia

Preguntas Adquirencia - EVA evento

- Garantizar que sean datáfonos de bancolombia: basta con la trxs?
- Garantizar que sean compras en Bogotá: basta con que lo hayan inscrito?
- Garantizar el valor total de la trx: mnt_base_trx, mnt_total_trx?
- Cual estado trx? Elegir: Cleared
- Hay compras en usd? En el periodo de tiempo las trxs son en COPS

In [7]:
# Transacciones con datafonos [adquirencia] desde el 20250911 hasta el 20250914
# 709211 trxs
sql_drop = """DROP TABLE IF EXISTS proceso_bluekai.mdo_adquirencia_trxs_20250911_20250914 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_bluekai.mdo_adquirencia_trxs_20250911_20250914 STORED AS PARQUET AS
SELECT cod_unico,
       nombre_comercio,
       pais_comercio,
       ciudad_comercio,
       area_evento,
       num_trj,
       tipo_tarjeta,
       metodo_captura,
       id_terminal,
       f_trx,
       id_cta,
       estado_trx,
       moneda_trx,
       mnt_base_trx,
       mnt_total_trx
FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
WHERE YEAR = 2025
  AND MONTH = 9
  AND DAY BETWEEN 1 AND 30
  AND tipo_trx = "Purchase"
  AND estado_trx = 'Cleared'
  AND (f_trx BETWEEN '2025-09-11 10:00:00' AND '2025-09-11 20:00:00'
       OR f_trx BETWEEN '2025-09-12 10:00:00' AND '2025-09-12 21:00:00'
       OR f_trx BETWEEN '2025-09-13 10:00:00' AND '2025-09-13 21:00:00'
       OR f_trx BETWEEN '2025-09-14 10:00:00' AND '2025-09-14 20:00:00');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso_bluekai.mdo_adquirencia_trxs_20250911_20250914;"""
helper.ejecutar_consulta(sql_compute)

-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 10/10    DROP ...do_adquirencia_trxs_20250911_20250914   finalizado   04:29:02 PM     00:00.2 
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 11/11  CREATE ...do_adquirencia_trxs_20250911_20250914   finalizado   04:29:02 PM     00:05.4 
-----------------------------------------------------------------------------------------------
----------------------------------------

# Obtener

# Análisis ingestión compras tabla transaccional adquirencia

In [ ]:
# Se deben esperar dos días para que se ingeste la información completa de las transacciones de un día en particular
# Para obtener la información de las compras del día jueves y viernes se debe espear hasta la sgte semana
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          left(cast(f_trx as string), 10) as f_trx,
          count(*) AS num_compras
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
   WHERE YEAR IN (2025)
     AND MONTH IN (9)
     AND DAY BETWEEN 1 AND 31
     and tipo_trx = "Purchase"
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
helper.obtener_dataframe(sql).head(60)

--------------------------------------------------------------
  i     tipo     nombre    estado     hora_inicio   duracion   
--------------------------------------------------------------
 2/2   DATAFRAME          falló n.1   10:41:50 AM             
 2/2   DATAFRAME        descargando   10:41:50 AM             

2025-09-19 10:41:59 - [INFO] - 514 filas, 10 columnas, 00:08.7 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 2/2   DATAFRAME         finalizado   10:41:50 AM     00:09.2 
--------------------------------------------------------------


,year,month,day,f_trx,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
0,2025,9,11,2025-10-01,1,1,1,1,1.0000,1.0000
1,2025,9,16,2025-09-19,1,1,1,1,1.0000,1.0000
2,2025,9,18,2025-09-18,59,59,59,1,1.0000,1.0000
3,2025,9,18,2025-09-17,1755424,1755448,1755448,2,1.0000,1.0000
4,2025,9,17,2025-09-17,24,1755448,24,1,0.0000,0.0000
5,2025,9,18,2025-09-16,40364,1777890,1777890,3,0.0227,1.0000
6,2025,9,17,2025-09-16,1737493,1777890,1737526,2,0.9773,0.9773
7,2025,9,16,2025-09-16,33,1777890,33,1,0.0000,0.0000
8,2025,9,18,2025-09-15,78,1731012,1731012,4,0.0000,1.0000
9,2025,9,17,2025-09-15,40834,1731012,1730934,3,0.0236,1.0000


In [5]:
helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_m_comercios')

2025-09-22 11:47:39 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_m_comercios
/Users/santlond/anaconda3/envs/env_odbc_py39/lib/python3.9/site-packages/helper/helper.py:421: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-09-22 11:47:40 - [INFO] - Finalizo la busqueda, duracion: 00:01.1, resultado: {'year': 2025, 'month': 9, 'day': 19}


{'year': 2025, 'month': 9, 'day': 19}